In [45]:
from ecmwf.opendata import Client

c = Client(source="ecmwf")

result = c.retrieve(
    step=list(range(0, 145, 3)),
    type="fc",
    param="ssr",
    target='data.grib2',
    date='2025-08-17',
    time=0,
)

print(result.datetime)

2025-08-17 00:00:00


In [46]:
%matplotlib inline

import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from matplotlib.animation import FuncAnimation
import cartopy.feature as cfeature
from IPython.display import HTML

# Load GRIB2 data
ds = xr.open_dataset("data.grib2", engine="cfgrib", decode_timedelta=True)

# Determine time dimension name
if 'step' in ds['ssr'].dims:
    time_dim = 'step'
elif 'time' in ds['ssr'].dims:
    time_dim = 'time'
else:
    raise ValueError("No time or step dimension found in 'ssr' variable.")

ssr = ds['ssr']
ssr = ssr.sel(latitude=slice(53.7, 50.5), longitude=slice(3.3, 7.2))

with plt.ioff():
    # Set up the plot
    fig, ax = plt.subplots(figsize=(10, 6), subplot_kw={'projection': ccrs.PlateCarree()})
    ax.set_global()

# Initial image (dummy)
img = ssr.isel({time_dim: 0}).plot(
    ax=ax, transform=ccrs.PlateCarree(), cmap='viridis', add_colorbar=True
)

# Update function
def update(frame):
    ax.clear()
    ax.set_extent([3.3, 7.2, 50.5, 53.7], crs=ccrs.PlateCarree())
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    data = ssr.isel({time_dim: frame})
    img = data.plot(
        ax=ax, transform=ccrs.PlateCarree(), cmap='viridis', add_colorbar=False
    )
    timestamp = str(data[time_dim].values)
    ax.set_title(f"Surface Solar Radiation - {timestamp}")
    return img,

# Create animation
ani = FuncAnimation(fig, update, frames=ssr[time_dim].size, interval=500, blit=False)

# Display it in Jupyter
HTML(ani.to_jshtml())


Ignoring index file 'data.grib2.5b7b6.idx' older than GRIB file


In [39]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

# Open the GRIB2 file with cfgrib engine
ds = xr.open_dataset("data.grib2", engine="cfgrib", decode_timedelta=True)

# Check the contents of the dataset
print(ds)

# Select the variable to plot (e.g., 'msl')
msl = ds['ssr'].isel(time=0)  # Select the first time step if 'time' is a dimension

# Plot
plt.figure(figsize=(10, 6))
ax = plt.axes(projection=ccrs.PlateCarree())
msl.plot(ax=ax, transform=ccrs.PlateCarree(), cmap='viridis')
ax.coastlines()
plt.title("Surface Solar Radiation")
plt.show()

Ignoring index file 'data.grib2.5b7b6.idx' older than GRIB file


<xarray.Dataset> Size: 37MB
Dimensions:     (step: 9, latitude: 721, longitude: 1440)
Coordinates:
    time        datetime64[ns] 8B ...
  * step        (step) timedelta64[ns] 72B 00:00:00 03:00:00 ... 1 days 00:00:00
    surface     float64 8B ...
  * latitude    (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude   (longitude) float64 12kB -180.0 -179.8 -179.5 ... 179.5 179.8
    valid_time  (step) datetime64[ns] 72B ...
Data variables:
    ssr         (step, latitude, longitude) float32 37MB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-08-18T08:07 GRIB to CDM+CF via cfgrib-0.9.1...


ValueError: Dimensions {'time'} do not exist. Expected one or more of ('step', 'latitude', 'longitude')